In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)

# ============================================================
# Configuration
# ============================================================

SEED = 42
DATA_FILE = "train.csv"
MODEL_NAME = "roberta-base"
OUTPUT_DIR = "./roberta_emotion_results"

MAX_INPUT_LENGTH = 128
NUM_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 4

# Hyperparameter search
HYPERPARAMETER_CONFIGS = [
    dict(
        learning_rate=1e-5,
        batch_size=16,
        weight_decay=0.01,
        warmup_ratio=0.10,
        gradient_accumulation_steps=1
    ),
    dict(
        learning_rate=2e-5,
        batch_size=16,
        weight_decay=0.01,
        warmup_ratio=0.10,
        gradient_accumulation_steps=1
    ),
    dict(
        learning_rate=3e-5,
        batch_size=16,
        weight_decay=0.01,
        warmup_ratio=0.10,
        gradient_accumulation_steps=1
    ),
    dict(
        learning_rate=2e-5,
        batch_size=8,
        weight_decay=0.05,
        warmup_ratio=0.10,
        gradient_accumulation_steps=2
    ),
    dict(
        learning_rate=1e-5,
        batch_size=8,
        weight_decay=0.05,
        warmup_ratio=0.15,
        gradient_accumulation_steps=2
    ),
]

# ============================================================
# Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

# ============================================================
# Load and preprocess data
# ============================================================

df = pd.read_csv(DATA_FILE)

if not {"text", "labels"}.issubset(df.columns):
    raise ValueError(
        "CSV must contain 'text' and 'labels' columns."
    )


def normalize_labels(x):
    """
    Convert different label formats into a clean list.
    """

    if isinstance(x, list):
        vals = x
    else:
        x = str(x).strip()

        if not x or x.lower() in {"nan", "none", "[]"}:
            return []

        # Remove brackets and quotes
        x = re.sub(r"[\[\]'\"]", "", x)

        # Normalize separators
        x = x.replace(",", ";")

        vals = x.split(";")

    return sorted(
        set(
            str(v).strip().lower()
            for v in vals
            if str(v).strip()
        )
    )


df["text"] = df["text"].fillna("").astype(str)

df["labels_list"] = df["labels"].apply(normalize_labels)

# Remove empty tweets
df = df[
    df["text"].str.strip().ne("")
].reset_index(drop=True)

# ============================================================
# Convert emotion labels to multi-hot vectors
# ============================================================

mlb = MultiLabelBinarizer()

label_matrix = mlb.fit_transform(df["labels_list"])

label_names = list(mlb.classes_)
NUM_LABELS = len(label_names)

print("\nEmotion labels:")
print(label_names)

print(f"\nNumber of emotion classes: {NUM_LABELS}")

# Store labels as lists of integers
df["label_vector"] = label_matrix.tolist()

# ============================================================
# Create Hugging Face Dataset
# ============================================================

dataset = Dataset.from_pandas(
    df[["text", "label_vector"]],
    preserve_index=False
)

# ============================================================
# 80/10/10 train-validation-test split
# ============================================================

split = dataset.train_test_split(
    test_size=0.20,
    seed=SEED
)

train_ds = split["train"]
tmp_ds = split["test"]

split2 = tmp_ds.train_test_split(
    test_size=0.50,
    seed=SEED
)

valid_ds = split2["train"]
test_ds = split2["test"]

print(
    f"\nTrain: {len(train_ds)} | "
    f"Validation: {len(valid_ds)} | "
    f"Test: {len(test_ds)}"
)

# ============================================================
# Tokenizer
# ============================================================

tokenizer = RobertaTokenizer.from_pretrained(
    MODEL_NAME
)


def tokenize(batch):
    """
    Tokenize tweets for RoBERTa.
    """

    encoded = tokenizer(
        batch["text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )

    # Cast labels to float32 for multi-label classification
    encoded["labels"] = [torch.tensor(label, dtype=torch.float32) for label in batch["label_vector"]]

    return encoded


train_tok = train_ds.map(
    tokenize,
    batched=True,
    remove_columns=train_ds.column_names
)

valid_tok = valid_ds.map(
    tokenize,
    batched=True,
    remove_columns=valid_ds.column_names
)

test_tok = test_ds.map(
    tokenize,
    batched=True,
    remove_columns=test_ds.column_names
)

# ============================================================
# Data collator
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest"
)

# ============================================================
# Model
# ============================================================

def make_model():
    """
    RoBERTa with a multi-label classification head.
    """

    model = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification"
    )

    return model


# ============================================================
# Metrics
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    # Convert logits to probabilities
    probabilities = 1 / (1 + np.exp(-logits))

    # Default classification threshold
    predictions = (probabilities >= 0.5).astype(int)

    labels = np.asarray(labels).astype(int)

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),

        "precision_micro": precision_score(
            labels,
            predictions,
            average="micro",
            zero_division=0
        ),

        "recall_micro": recall_score(
            labels,
            predictions,
            average="micro",
            zero_division=0
        ),

        "f1_micro": f1_score(
            labels,
            predictions,
            average="micro",
            zero_division=0
        )
    }


# ============================================================
# One hyperparameter trial
# ============================================================

def run_trial(config, trial):

    trial_dir = os.path.join(
        OUTPUT_DIR,
        f"trial_{trial}"
    )

    os.makedirs(
        trial_dir,
        exist_ok=True
    )

    model = make_model()

    # --------------------------------------------------------
    # Calculate warm-up steps
    # --------------------------------------------------------

    total_train_samples = len(train_tok)

    num_update_steps_per_epoch = (
        total_train_samples
        // (
            config["batch_size"]
            * config["gradient_accumulation_steps"]
        )
    )

    if total_train_samples % (
        config["batch_size"]
        * config["gradient_accumulation_steps"]
    ) != 0:

        num_update_steps_per_epoch += 1

    total_training_steps = (
        num_update_steps_per_epoch
        * NUM_EPOCHS
    )

    warmup_steps = int(
        total_training_steps
        * config["warmup_ratio"]
    )

    # --------------------------------------------------------
    # Training arguments
    # --------------------------------------------------------

    args = TrainingArguments(
        output_dir=trial_dir,

        learning_rate=config["learning_rate"],

        per_device_train_batch_size=
            config["batch_size"],

        per_device_eval_batch_size=
            config["batch_size"],

        gradient_accumulation_steps=
            config["gradient_accumulation_steps"],

        weight_decay=config["weight_decay"],

        warmup_steps=warmup_steps,

        max_grad_norm=1.0,

        num_train_epochs=NUM_EPOCHS,

        eval_strategy="epoch",

        save_strategy="epoch",

        logging_strategy="epoch",

        load_best_model_at_end=True,

        metric_for_best_model="f1_micro",

        greater_is_better=True,

        fp16=torch.cuda.is_available(),

        seed=SEED,

        data_seed=SEED,

        report_to="none",

        save_total_limit=2
    )

    # --------------------------------------------------------
    # Trainer
    # --------------------------------------------------------

    trainer = Trainer(
        model=model,

        args=args,

        train_dataset=train_tok,

        eval_dataset=valid_tok,

        data_collator=data_collator,

        compute_metrics=compute_metrics,

        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=
                    EARLY_STOPPING_PATIENCE
            )
        ]
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    trainer.train()

    # --------------------------------------------------------
    # Validation evaluation
    # --------------------------------------------------------

    metrics = trainer.evaluate(
        metric_key_prefix="validation"
    )

    result = {
        "trial": trial,

        **config,

        "validation_accuracy":
            metrics.get(
                "validation_accuracy",
                0
            ),

        "validation_precision_micro":
            metrics.get(
                "validation_precision_micro",
                0
            ),

        "validation_recall_micro":
            metrics.get(
                "validation_recall_micro",
                0
            ),

        "validation_f1_micro":
            metrics.get(
                "validation_f1_micro",
                0
            )
    }

    # Save best checkpoint
    trainer.save_model(
        os.path.join(
            trial_dir,
            "best_model"
        )
    )

    tokenizer.save_pretrained(
        os.path.join(
            trial_dir,
            "best_model"
        )
    )

    return result


# ============================================================
# Hyperparameter search
# ============================================================

results = []

for i, config in enumerate(
    HYPERPARAMETER_CONFIGS,
    1
):

    print("\n" + "=" * 60)

    print(
        f"Trial {i}: {config}"
    )

    result = run_trial(
        config,
        i
    )

    results.append(result)

    print(
        "Validation F1:",
        round(
            result["validation_f1_micro"],
            4
        )
    )


# ============================================================
# Save hyperparameter results
# ============================================================

results_df = pd.DataFrame(
    results
).sort_values(
    "validation_f1_micro",
    ascending=False
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "hyperparameter_search_results.csv"
    ),
    index=False
)

# ============================================================
# Select best model
# ============================================================

best = results_df.iloc[0]

best_trial = int(
    best["trial"]
)

best_path = os.path.join(
    OUTPUT_DIR,
    f"trial_{best_trial}",
    "best_model"
)

print("\n" + "=" * 60)
print("BEST CONFIGURATION")
print("=" * 60)

print(
    best.to_dict()
)

# ============================================================
# Final test evaluation
# ============================================================

final_model = (
    RobertaForSequenceClassification
    .from_pretrained(best_path)
)

final_args = TrainingArguments(
    output_dir=os.path.join(
        OUTPUT_DIR,
        "final"
    ),

    per_device_eval_batch_size=
        int(best["batch_size"]),

    fp16=torch.cuda.is_available(),

    report_to="none"
)

final_trainer = Trainer(
    model=final_model,

    args=final_args,

    eval_dataset=test_tok,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# ============================================================
# Test evaluation
# ============================================================

test_metrics = final_trainer.evaluate(
    metric_key_prefix="test"
)

print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(
    f"Accuracy:        "
    f"{test_metrics.get('test_accuracy', 0):.4f}"
)

print(
    f"Precision micro: "
    f"{test_metrics.get('test_precision_micro', 0):.4f}"
)

print(
    f"Recall micro:    "
    f"{test_metrics.get('test_recall_micro', 0):.4f}"
)

print(
    f"F1 micro:        "
    f"{test_metrics.get('test_f1_micro', 0):.4f}"
)

# ============================================================
# Save final metrics
# ============================================================

pd.DataFrame(
    [test_metrics]
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_test_metrics.csv"
    ),
    index=False
)

# ============================================================
# Save emotion label mapping
# ============================================================

label_mapping = pd.DataFrame({
    "label_id": range(NUM_LABELS),
    "emotion": label_names
})

label_mapping.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "emotion_label_mapping.csv"
    ),
    index=False
)

print("\nEmotion label mapping:")
print(label_mapping)


Emotion labels:
['1', '2', '3', '4', '5', '6', '7', '8']

Number of emotion classes: 8

Train: 838857 | Validation: 104857 | Test: 104858


Map:   0%|          | 0/838857 [00:00<?, ? examples/s]

Map:   0%|          | 0/104857 [00:00<?, ? examples/s]

Map:   0%|          | 0/104858 [00:00<?, ? examples/s]


Trial 1: {'learning_rate': 1e-05, 'batch_size': 16, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'gradient_accumulation_steps': 1}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  su

Epoch,Training Loss,Validation Loss
